# Dartboard RAG：具有平衡相关性和多样性的检索增强生成

## 概述
**Dartboard RAG** 流程解决了大型知识库中的一个常见挑战：确保检索到的信息既相关又非冗余。通过显式优化组合的相关性-多样性评分函数，它可以防止多个 top-k 文档提供相同的信息。这种方法源自论文中的优雅方法：

> [*使用相关信息增益获得更好的 RAG*](https://arxiv.org/abs/2407.12101)

该论文概述了核心思想的三种变体——混合 RAG（密集 + 稀疏）、交叉编码器版本和普通方法。 **普通方法**最直接地传达了基本概念，并且此实现通过可选权重对其进行了扩展，以控制相关性和多样性之间的平衡。

## 动机

1. **密集、重叠的知识库**  
   在大型数据库中，文档可能会重复相似的内容，从而导致 top-k 检索中出现冗余。

2. **提高信息覆盖范围**  
   将相关性和多样性相结合可以产生更丰富的文档集，从而减轻内容过于相似的“回音室”效应。


## 关键组件

1. **相关性和多样性的结合**  
   - 计算分数，考虑文档与查询的相关程度以及它与已选择文档的区别程度。

2. **加权平衡**  
   - 引入 RELEVANCE_WEIGHT 和 DIVERSITY_WEIGHT 以允许动态控制评分。  
   - 有助于避免过于多样化但相关性较低的结果。

3. **生产就绪代码**  
   - 源自官方实施，但为了清晰起见进行了重新组织。  
   - 可以更轻松地集成到现有的 RAG 管道中。

## 方法详细信息

1. **文献检索**  
   - 根据相似性（例如余弦或 BM25）获取初始候选文档集。  
   - 通常检索前 N 个候选者作为起点。

2. **评分与选择**  
   - 每个文档的总体得分结合了**相关性**和**多样性**：  
   - 选择得分最高的文档，然后对与其过于相似的文档进行惩罚。  
   - 重复直到识别出 top-k 文档。

3. **混合/融合和交叉编码器支持**  
   本质上，您所需要的只是文档与查询之间的距离以及文档之间的距离。您可以轻松地从混合/融合检索或跨编码器检索中提取这些内容。我唯一的建议是少依赖基于排名的分数。
   - 对于**混合/融合检索**：将相似性（密集和稀疏/BM25）合并为单个距离。这可以通过结合密集向量和稀疏向量的余弦相似度（例如对它们求平均值）来实现。移动到距离很简单（1 - 平均余弦相似度）。 
   - 对于**交叉编码器**：您可以直接使用交叉编码器相似度分数（1-相似度），可能会根据缩放因子进行调整。

4. **平衡与调整**  
   - 根据您的需求和数据集的密度调整 DIVERSITY_WEIGHT 和 RELEVANCE_WEIGHT。  



通过将**相关性**和**多样性**集成到检索中，Dartboard RAG 方法可确保 top-k 文档共同提供更丰富、更全面的信息，从而在检索增强生成系统中实现更高质量的响应。

该论文还有一个官方代码实现，该代码基于它，但我认为这里的代码更具可读性、可管理性和生产就绪性。

# 包安装和导入

下面的单元格安装运行此笔记本所需的所有必需软件包。

In [ ]:
# Install required packages
!pip install numpy python-dotenv

In [ ]:
# Clone the repository to access helper functions and evaluation modules
!git clone https://github.com/NirDiamant/RAG_TECHNIQUES.git
import sys
sys.path.append('RAG_TECHNIQUES')
# If you need to run with the latest data
# !cp -r RAG_TECHNIQUES/data .

In [ ]:
import os
import sys
from dotenv import load_dotenv
from scipy.special import logsumexp
from typing import Tuple, List, Any
import numpy as np

# Load environment variables from a .env file
load_dotenv()
# Set the OpenAI API key environment variable (comment out if not using OpenAI)
if not os.getenv('OPENAI_API_KEY'):
    print("Please enter your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = input("Please enter your OpenAI API key: ")
else:
    os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

# Original path append replaced for Colab compatibility
from helper_functions import *
from evaluation.evalute_rag import *


Please enter your OpenAI API key: 


### 阅读文档

In [ ]:
# Download required data files
import os
os.makedirs('data', exist_ok=True)

# Download the PDF document used in this notebook
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf


In [3]:
path = "data/Understanding_Climate_Change.pdf"

### 编码文档

In [4]:
# this part is same like simple_rag.ipynb, only simulating a dense dataset
def encode_pdf(path, chunk_size=1000, chunk_overlap=200):
    """
    Encodes a PDF book into a vector store using OpenAI embeddings.

    Args:
        path: The path to the PDF file.
        chunk_size: The desired size of each text chunk.
        chunk_overlap: The amount of overlap between consecutive chunks.

    Returns:
        A FAISS vector store containing the encoded book content.
    """

    # Load PDF documents
    loader = PyPDFLoader(path)
    documents = loader.load()
    documents=documents*5 # load every document 5 times to emulate a dense dataset

    # Split documents into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len
    )
    texts = text_splitter.split_documents(documents)
    cleaned_texts = replace_t_with_space(texts)

    # Create embeddings (Tested with OpenAI and Amazon Bedrock)
    embeddings = get_langchain_embedding_provider(EmbeddingProvider.OPENAI)
    #embeddings = get_langchain_embedding_provider(EmbeddingProvider.AMAZON_BEDROCK)

    # Create vector store
    vectorstore = FAISS.from_documents(cleaned_texts, embeddings)

    return vectorstore

### 创建支持存储

In [5]:
chunks_vector_store = encode_pdf(path, chunk_size=1000, chunk_overlap=200)

### 使用支持存储进行检索的一些辅助函数。
这部分与 simple_rag.ipynb 相同，只是它使用实际的 FAISS 索引（而不是包装器）

In [6]:

def idx_to_text(idx:int):
    """
    Convert a Vector store index to the corresponding text.
    """
    docstore_id = chunks_vector_store.index_to_docstore_id[idx]
    document = chunks_vector_store.docstore.search(docstore_id)
    return document.page_content


def get_context(query:str,k:int=5) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """
    Retrieve top k context items for a query using top k retrieval.
    """
    # regular top k retrieval
    q_vec=chunks_vector_store.embedding_function.embed_documents([query])
    _,indices=chunks_vector_store.index.search(np.array(q_vec),k=k)

    texts = [idx_to_text(i) for i in indices[0]]
    return texts


In [10]:

test_query = "What is the main cause of climate change?"


### 常规top k检索
- 这个演示表明，当数据库密集时（这里我们通过加载每个文档 5 次来模拟密度），结果不好，我们没有得到最相关的结果。请注意，前 3 个结果都是同一文档的重复项。

In [11]:
texts=get_context(test_query,k=3)
show_context(texts)

Context 1:
driven by human activities, particularly the emission of greenhou se gases.  
Chapter 2: Causes of Climate Change  
Greenhouse Gases  
The primary cause of recent climate change is the increase in greenhouse gases in the 
atmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous 
oxide (N2O), trap heat from the sun, creating a "greenhouse effect." This effect is  essential 
for life on Earth, as it keeps the planet warm enough to support life. However, human 
activities have intensified this natural process, leading to a warmer climate.  
Fossil Fuels  
Burning fossil fuels for energy releases large amounts of CO2. This includes coal, oil, and 
natural gas used for electricity, heating, and transportation. The industrial revolution marked 
the beginning of a significant increase in fossil fuel consumption, which continues to rise 
today.  
Coal


Context 2:
driven by human activities, particularly the emission of greenhou se gases.  
Chapter 2: C

## 现在是真正的部分:)

### 更多用于距离归一化的实用程序

In [21]:
def lognorm(dist:np.ndarray, sigma:float):
    """
    Calculate the log-normal probability for a given distance and sigma.
    """
    if sigma < 1e-9: 
        return -np.inf * dist
    return -np.log(sigma) - 0.5 * np.log(2 * np.pi) - dist**2 / (2 * sigma**2)


## 贪心飞镖搜索

这是核心算法：一种搜索算法，通过平衡两个因素从集合中选择一组不同的相关文档：与查询的相关性和所选文档之间的多样性。

给定查询和文档之间的距离，加上所有文档之间的距离，算法：

1. 首先选择最相关的文档
2. 通过组合以下方式迭代选择其他文档：
   - 与原始查询的相关性
   - 与之前选定的文件不同

相关性和多样性之间的平衡由权重控制：
- `DIVERSITY_WEIGHT`：与现有选择的差异的重要性
- `RELEVANCE_WEIGHT`：相关性对查询的重要性
- `SIGMA`：概率转换的平滑参数

该算法会返回选定的文档及其选择分数，这对于您需要相关但多样化的结果的搜索结果等应用程序非常有用。

例如，在搜索新闻文章时，它会首先返回最相关的文章，然后查找既切题又提供新信息的文章，避免冗余选择。

In [ ]:
# Configuration parameters
DIVERSITY_WEIGHT = 1.0  # Weight for diversity in document selection
RELEVANCE_WEIGHT = 1.0  # Weight for relevance to query
SIGMA = 0.1  # Smoothing parameter for probability distribution

def greedy_dartsearch(
    query_distances: np.ndarray,
    document_distances: np.ndarray,
    documents: List[str],
    num_results: int
) -> Tuple[List[str], List[float]]:
    """
    Perform greedy dartboard search to select top k documents balancing relevance and diversity.
    
    Args:
        query_distances: Distance between query and each document
        document_distances: Pairwise distances between documents
        documents: List of document texts
        num_results: Number of documents to return
    
    Returns:
        Tuple containing:
        - List of selected document texts
        - List of selection scores for each document
    """
    # Avoid division by zero in probability calculations
    sigma = max(SIGMA, 1e-5)
    
    # Convert distances to probability distributions
    query_probabilities = lognorm(query_distances, sigma)
    document_probabilities = lognorm(document_distances, sigma)
    
    # Initialize with most relevant document
    
    most_relevant_idx = np.argmax(query_probabilities)
    selected_indices = np.array([most_relevant_idx])
    selection_scores = [1.0] # dummy score for the first document
    # Get initial distances from the first selected document
    max_distances = document_probabilities[most_relevant_idx]
    
    # Select remaining documents
    while len(selected_indices) < num_results:
        # Update maximum distances considering new document
        updated_distances = np.maximum(max_distances, document_probabilities)
        
        # Calculate combined diversity and relevance scores
        combined_scores = (
            updated_distances * DIVERSITY_WEIGHT +
            query_probabilities * RELEVANCE_WEIGHT
        )
        
        # Normalize scores and mask already selected documents
        normalized_scores = logsumexp(combined_scores, axis=1)
        normalized_scores[selected_indices] = -np.inf
        
        # Select best remaining document
        best_idx = np.argmax(normalized_scores)
        best_score = np.max(normalized_scores)
        
        # Update tracking variables
        max_distances = updated_distances[best_idx]
        selected_indices = np.append(selected_indices, best_idx)
        selection_scores.append(best_score)
    
    # Return selected documents and their scores
    selected_documents = [documents[i] for i in selected_indices]
    return selected_documents, selection_scores

## 飞镖上下文检索

### 使用飞镖盘检索的主要功能。它代替了 get_context（这是简单的 RAG）。它：

1. 接受文本查询，对其进行向量化，通过简单的 RAG 获取前 k 个文档（及其向量）
2. 使用这些向量来计算查询和候选匹配之间的相似度
3. 运行 dartboard 算法将候选匹配细化为 k 个文档的最终列表
4. 返回最终的文档列表及其分数

In [ ]:

def get_context_with_dartboard(
    query: str,
    num_results: int = 5,
    oversampling_factor: int = 3
) -> Tuple[List[str], List[float]]:
    """
    Retrieve most relevant and diverse context items for a query using the dartboard algorithm.
    
    Args:
        query: The search query string
        num_results: Number of context items to return (default: 5)
        oversampling_factor: Factor to oversample initial results for better diversity (default: 3)
    
    Returns:
        Tuple containing:
        - List of selected context texts
        - List of selection scores
        
    Note:
        The function uses cosine similarity converted to distance. Initial retrieval 
        fetches oversampling_factor * num_results items to ensure sufficient diversity 
        in the final selection.
    """
    # Embed query and retrieve initial candidates
    query_embedding = chunks_vector_store.embedding_function.embed_documents([query])
    _, candidate_indices = chunks_vector_store.index.search(
        np.array(query_embedding),
        k=num_results * oversampling_factor
    )
    
    # Get document vectors and texts for candidates
    candidate_vectors = np.array(
        chunks_vector_store.index.reconstruct_batch(candidate_indices[0])
    )
    candidate_texts = [idx_to_text(idx) for idx in candidate_indices[0]]
    
    # Calculate distance matrices
    # Using 1 - cosine_similarity as distance metric
    document_distances = 1 - np.dot(candidate_vectors, candidate_vectors.T)
    query_distances = 1 - np.dot(query_embedding, candidate_vectors.T)
    
    # Apply dartboard selection algorithm
    selected_texts, selection_scores = greedy_dartsearch(
        query_distances,
        document_distances,
        candidate_texts,
        num_results
    )
    
    return selected_texts, selection_scores

### 飞镖检索 - 相同查询、k 和数据集的结果
- 正如您现在所看到的，前 3 个结果不仅仅是重复。

In [22]:
texts,scores=get_context_with_dartboard(test_query,k=3)
show_context(texts)


Context 1:
driven by human activities, particularly the emission of greenhou se gases.  
Chapter 2: Causes of Climate Change  
Greenhouse Gases  
The primary cause of recent climate change is the increase in greenhouse gases in the 
atmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous 
oxide (N2O), trap heat from the sun, creating a "greenhouse effect." This effect is  essential 
for life on Earth, as it keeps the planet warm enough to support life. However, human 
activities have intensified this natural process, leading to a warmer climate.  
Fossil Fuels  
Burning fossil fuels for energy releases large amounts of CO2. This includes coal, oil, and 
natural gas used for electricity, heating, and transportation. The industrial revolution marked 
the beginning of a significant increase in fossil fuel consumption, which continues to rise 
today.  
Coal


Context 2:
Most of these climate changes are attributed to very small variations in Earth's orbit tha

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--dartboard)